# 07 — Three-Engine Evidence Tree & Explain Demo

One schema, three engines, shared Store. All output from real framework calls.

**Demonstrates:**
1. **Evidence tree** — full node hierarchy with multi-rule chains (recursive_depth > 0)
2. **Certainty** — condition_weights + confidence → bottleneck vs additive
3. **ProbLog proof tree** — deep proof_goal/proof_leaf hierarchy + probability pipeline
4. **PyReason timeline** — multi-node graph propagation across timesteps, multiple chains
5. **Cross-engine data flow** — accepted facts from one engine feed the next
6. **Full explain pipeline** — tree/timeline → summary → narrative → NL → **steps** → HTML for each engine

ProbLog and PyReason use mocked runners; the entire evaluate → accept → explain pipeline is real.

## 0. Imports

In [1]:
import sys, tempfile
from pathlib import Path
from pprint import pprint
from unittest.mock import patch

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from kernel.sdk import SDKStore, Entity, Identity, Field, Relationship, Rule, Pred, vars as sdk_vars
from kernel.sdk.compile import compile_schema_from_classes
from kernel.sdk.dsl.rule import RuleRef
from kernel.authoring import FileAuthoringRegistry
from service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation, accept_runtime_derivation,
    explain_runtime_tree, explain_runtime_summary, explain_runtime_narrative, explain_runtime_nl,
    explain_runtime_timeline, explain_runtime_timeline_summary, explain_runtime_timeline_narrative,
    explain_runtime_steps,
    register_ephemeral_rule, clear_ephemeral_rules,
    get_runtime_session_rules, list_runtime_candidates,
)
from kernel.audit.evidence_graph import render_evidence_graph_html
from service.static_ui import render_candidate_evidence_html
import kernel.adapters.problog
import kernel.adapters.pyreason
from kernel.adapters.pyreason.session import PyReasonSession
from kernel.adapters.pyreason.runner import PyReasonRunConfig, PyReasonRunResult
from IPython.display import HTML, display

In [2]:
def print_tree(node, indent=0):
    prefix = "  " * indent
    kind = node.get("node_kind", "?")
    parts = [kind]
    if kind == "candidate_result":
        parts.append(f"root_result_kind={node.get('root_result_kind', '?')}")
        em = node.get("engine_meta")
        if em: parts.append(f"engine_meta={em}")
    elif kind == "predicate_witness_group":
        parts.append(f"pred_id={node.get('pred_id', '?')}")
        cc = node.get("condition_confidence")
        if cc is not None: parts.append(f"cc={cc}")
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        parts.append(f"[{', '.join(c.get('val','?') for c in claims)}]")
        conf = node.get("confidence")
        if conf is not None: parts.append(f"conf={conf}")
    elif kind in ("proof_goal", "proof_leaf"):
        parts.append(f"pred={node.get('pred_id', '?')}")
        args = node.get("goal_args", [])
        if args: parts.append(f"args={args[:2]}{'...' if len(args)>2 else ''}")
    elif kind == "rule_ref":
        parts.append(f"{node.get('rule_ref_id', '?')} v{node.get('rule_ref_version', '?')}")
    elif kind == "non_fact_check":
        parts.append(f"{node.get('check_kind', '?')} status={node.get('status', '?')}")
    print(f"{prefix}- {' | '.join(parts)}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

## 1. Schema + Session

Research collaboration network with `Researcher` nodes and `Collaboration` edges.

The next cell seeds three researchers reused by all later sections:
- two satisfy the native grant rule threshold (`publications = 50+`)
- one is intentionally below threshold (`publications = 30`)
- the same refs later feed the mocked ProbLog proof trace and PyReason timeline

All printed output below is derived from runtime responses or seeded Python objects, not narrated by fixed scene text.


In [3]:
class Researcher(Entity):
    researcher_id: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")
    expertise: str = Field(cardinality="single")
    h_index: str = Field(cardinality="single")
    publications: str = Field(cardinality="single")
    is_established: str = Field(cardinality="single")
    qualifies_grant: str = Field(cardinality="single")
    tag_seed: str = Field(cardinality="single")
    tag: str = Field(cardinality="single")
    risk_flag: str = Field(cardinality="single")

class Collaboration(Relationship):
    from_entity = Researcher
    to_entity = Researcher
    joint_papers: str = Field(cardinality="single")

schema_ir = compile_schema_from_classes([Researcher, Collaboration])
sdk = SDKStore([Researcher], schema_ir=schema_ir)

# ── Multi-rule chain: established_check → grant_qualification ──
with sdk_vars("r", "exp", "h") as (r, exp, h):
    established_check = Rule(
        id="q.established_check", version="1.0.0",
        select=[r, exp],
        where=[Pred("researcher:expertise", r, exp),
               Pred("researcher:h_index", r, h),
               Pred("researcher:publications", r, "50+")],
        expose=True,
        condition_weights={"b0.a0": 0.7, "b0.a1": 0.9, "b0.a2": 0.5},
    )

with sdk_vars("r", "exp", "pub") as (r, exp, pub):
    grant_rule = Rule(
        id="q.grant_qualification", version="1.0.0",
        select=[r, exp],
        where=[RuleRef(established_check)(r, exp),
               Pred("researcher:publications", r, pub)],
        expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a1": 0.6},
    )

# Registry + session
registry_dir = tempfile.mkdtemp(prefix="three_engine_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
for rule in [established_check, grant_rule]:
    registry.register_rule_spec(sdk._compile_rule_input(rule))

reset_runtime_sessions_for_tests()
resp = open_runtime_session({"registry_root": registry_dir})
session_id = resp["session"]["session_id"]

# Seed 3 researchers
alice_ref = sdk.ref(Researcher, researcher_id="Alice")
bob_ref = sdk.ref(Researcher, researcher_id="Bob")
carol_ref = sdk.ref(Researcher, researcher_id="Carol")

researchers = [
    (alice_ref, "Alice Chen",   "NLP",      "42", "50+", 0.95, 0.88, 0.7),
    (bob_ref,   "Bob Zhang",    "CV",       "35", "50+", 0.90, 0.75, 0.6),
    (carol_ref, "Carol Li",     "RL",       "28", "30",  0.85, 0.65, 0.5),
]
for ref, name, exp, h_idx, pubs, exp_conf, h_conf, pub_conf in researchers:
    write_runtime_fact(session_id, {"pred_id": "researcher:name", "e_ref": ref,
        "rest_terms": [["string", name]]}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:expertise", "e_ref": ref,
        "rest_terms": [["string", exp]], "meta": {"confidence": exp_conf}}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:h_index", "e_ref": ref,
        "rest_terms": [["string", h_idx]], "meta": {"confidence": h_conf}}, kind="add")
    pub_meta: dict = {"confidence": pub_conf}
    if ref == alice_ref:
        pub_meta["source"] = "ORCID public database"
        pub_meta["approved_by"] = "data-pipeline-v2"
    write_runtime_fact(session_id, {"pred_id": "researcher:publications", "e_ref": ref,
        "rest_terms": [["string", pubs]], "meta": pub_meta}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:tag_seed", "e_ref": ref,
        "rest_terms": [["string", "senior"]], "meta": {"confidence": 1.0}}, kind="add")

print(f"Session ready: {len(researchers)} researchers seeded")
for _ref, name, exp, h_idx, pubs, exp_conf, h_conf, pub_conf in researchers:
    print(
        f"  {name}: expertise={exp}, h_index={h_idx}, publications={pubs}, "
        f"confidence=(expertise={exp_conf}, h_index={h_conf}, publications={pub_conf})"
    )


Session ready: 3 researchers seeded
  Alice Chen: expertise=NLP, h_index=42, publications=50+, confidence=(expertise=0.95, h_index=0.88, publications=0.7)
  Bob Zhang: expertise=CV, h_index=35, publications=50+, confidence=(expertise=0.9, h_index=0.75, publications=0.6)
  Carol Li: expertise=RL, h_index=28, publications=30, confidence=(expertise=0.85, h_index=0.65, publications=0.5)


---
## 2. Native Engine — Multi-Rule Evidence Tree with Certainty

`grant_qualification` chains through `established_check` via `RuleRef` → creates
a tree with `referenced_support` subtree and `recursive_depth > 0`.

Expected native outcome in this dataset:
- exactly two candidates should be accepted
- certainty is derived from the same tree through the eligible child-proof subtree

### 2.1 Evaluate + Accept


In [4]:
eval_native = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.grant", "version": "1.0.0",
    "target": "researcher:qualifies_grant", "head_vars": ["$r", "$exp"],
    "where": [["ruleref", "q.grant_qualification", "1.0.0", ["$r", "$exp"]]],
    "mode": "native",
}})

native_cands = eval_native["evaluation"]["candidates"]
accepted_native = []
for c in native_cands:
    accepted_native.append(accept_runtime_derivation(session_id, {"candidate": c}))

cid_native = native_cands[0]["candidate_id"]
print(f"Accepted native candidates: {len(native_cands)}")
print(f"  confidence_kinds: {sorted({c['confidence_kind'] for c in native_cands})}")
print(f"  first_candidate_id: {cid_native}")


Accepted native candidates: 2
  confidence_kinds: ['none']
  first_candidate_id: cand_v2:d84734a81cbfb11a4292fcf6f0cb886a44c9035be3bc376a0a1deb92745b24f6


### 2.2 Evidence Tree — Multi-Rule Node Hierarchy

Shows the full chain: `candidate_result` → `rule_ref(grant_qualification)` →
`referenced_support` → `rule_ref(established_check)` → witness groups + assertions.

`recursive_depth` will be > 0 due to the chained `RuleRef`.

In [5]:
tree_resp = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_native})
print_tree(tree_resp["tree"]["root"])


- candidate_result | root_result_kind=fact
  - support_section
    - non_fact_check | ruleref status=satisfied
  - rule_ref_section
    - rule_ref | q.grant_qualification v1.0.0
      - referenced_support
        - support_section
          - predicate_witness_group | pred_id=researcher:publications | cc=0.7
            - assertion_fact | [50+] | conf=0.7
          - non_fact_check | ruleref status=satisfied
        - rule_ref_section
          - rule_ref | q.established_check v1.0.0
            - referenced_support
              - support_section
                - predicate_witness_group | pred_id=researcher:expertise | cc=0.95
                  - assertion_fact | [NLP] | conf=0.95
                - predicate_witness_group | pred_id=researcher:h_index | cc=0.88
                  - assertion_fact | [42] | conf=0.88
                - predicate_witness_group | pred_id=researcher:publications | cc=0.7
                  - assertion_fact | [50+] | conf=0.7


### 2.3 Certainty — Bottleneck vs Additive

In [6]:
s_bn = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
s_ad = explain_runtime_summary(session_id, {
    "kind": "candidate", "id": cid_native, "certainty_aggregation": "additive"})
cs_bn, cs_ad = s_bn.get("certainty_summary"), s_ad.get("certainty_summary")

print(f"recursive_depth = {s_bn['summary']['recursive_depth']}")
print(f"witness_assertion_count = {s_bn['summary']['witness_assertion_count']}")
print(f"rule_ref_count = {s_bn['summary']['rule_ref_count']}")

if cs_bn and cs_ad:
    print(f"\n{'':25s} {'Bottleneck':>12s}  {'Additive':>12s}")
    print(f"{'Aggregate':25s} {cs_bn['aggregate_certainty']:>12}  {cs_ad['aggregate_certainty']:>12}")
    for b, a in zip(cs_bn["conditions"], cs_ad["conditions"]):
        print(f"  {b['atom_key']:23s} {b['impact']:>12}  {a['impact']:>12}")

recursive_depth = 2
witness_assertion_count = 4
rule_ref_count = 2


### 2.4 Narrative + NL

In [7]:
narr = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_native})
nl = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_native})

n = narr["narrative"]
print(f"Headline: {n['headline']}")
for section in ["overview_lines", "evidence_lines", "rule_chain_lines", "certainty_lines"]:
    lines = n.get(section, [])
    if lines:
        print(f"\n{section}:")
        for line in lines: print(f"  {line}")
bn = n.get("certainty_bottleneck")
if bn: print(f"  bottleneck: {bn}")

print(f"\nNL ({len(nl['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p[:150]}{'...' if len(p)>150 else ''}")

Headline: Candidate cand_v2:d84734a81cbfb11a4292fcf6f0cb886a44c9035be3bc376a0a1deb92745b24f6 uses support kind native_binding_v1 across 20 tree node(s).

overview_lines:
  Root result kind: fact.
  Role counts: structural=6, witness=8, constraint=2, rule_chain=4, terminal=0, degraded=0.
  Recursive depth: 2.

evidence_lines:
  Witness assertions: 4.
  Witness nodes: 8; constraint nodes: 2.

rule_chain_lines:
  Rule reference nodes: 2.
  Recursive proof depth: 2.

NL (5 paragraphs):
  [1] Candidate cand_v2:d84734a81cbfb11a4292fcf6f0cb886a44c9035be3bc376a0a1deb92745b24f6 uses support kind native_binding_v1 across 20 tree node(s). Root re...
  [2] Evidence summary: Witness assertions: 4. Witness nodes: 8; constraint nodes: 2.
  [3] Rule-chain summary: Rule reference nodes: 2. Recursive proof depth: 2.
  [4] Terminal and drill-down summary: No unresolved support or recursion boundaries were encountered. Open referenced support branches to inspect recursive...
  [5] Fact sources: researcher

### 2.5 Explain Steps — Fact-to-Conclusion Path

In [8]:
steps_n = explain_runtime_steps(session_id, {"kind": "candidate", "id": cid_native})
print(f"explain-steps: {len(steps_n['steps'])} steps ({steps_n.get('support_kind', '?')})")
for s in steps_n["steps"]:
    indent = "  " * s["detail"]["depth"]
    print(f"  {s['step_num']:2}. {indent}[{s['step_kind']}] {s['description']}")

explain-steps: 8 steps (?)
   1.   [rule_apply] Support group satisfied: 0 condition(s) met
   2.     [fact_check] Fact researcher:publications(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) = 50+ ✓
   3.   [rule_apply] Support group satisfied: 1 condition(s) met
   4.     [fact_check] Fact researcher:expertise(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) = NLP ✓
   5.     [fact_check] Fact researcher:h_index(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) = 42 ✓
   6.     [fact_check] Fact researcher:publications(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) = 50+ ✓
   7.   [rule_apply] Support group satisfied: 3 condition(s) met
   8. [conclusion] Candidate cand_v2:d84734a81cbfb11a4292fcf6f0cb886a44c9035be3bc376a0a1deb92745b24f6 established (fact)


In [9]:
narr_src = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_native})
source_lines = narr_src["narrative"].get("source_lines", [])
print("Fact provenance (source_lines):")
if source_lines:
    for line in source_lines:
        print(f"  {line}")
else:
    print("  (none)")

nl_src = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_native})
src_paras = [p for p in nl_src["explain_nl"]["paragraphs"] if p.startswith("Fact sources:")]
if src_paras:
    print(f"\nNL source paragraph: {src_paras[0]}")

Fact provenance (source_lines):
  researcher:publications(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) - from 'ORCID public database' (approved by data-pipeline-v2)
  researcher:publications(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) - from 'ORCID public database' (approved by data-pipeline-v2)

NL source paragraph: Fact sources: researcher:publications(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) - from 'ORCID public database' (approved by data-pipeline-v2) researcher:publications(idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq) - from 'ORCID public database' (approved by data-pipeline-v2)


### 2.6 HTML Rendering

In [25]:
html_native = render_candidate_evidence_html(tree_resp["tree"], narrative=narr.get("narrative"))
print(f"HTML: {len(html_native)} chars")
display(HTML(f"<div style='border:2px solid #3498db;border-radius:8px;overflow:hidden;margin:10px 0'>"
             f"<div style='background:#3498db;color:white;padding:6px 12px;font-size:13px'>"
             f"Native — Multi-Rule Evidence Tree + Certainty</div>"
             f"<iframe srcdoc=\'{html_native.replace(chr(39), '&#39;')}\' "
             f"style='width:100%;height:500px;border:none'></iframe></div>"))

HTML: 39684 chars


### 2.7 Session-Scoped Ephemeral Rule

In [ ]:
# Register an ephemeral rule into the live session (G4)
reg_resp = register_ephemeral_rule(session_id, {
    "rule": {
        "rule_id": "q.eph.expert_tag",
        "version": "v1",
        "select_vars": ["$r", "$exp"],
        "where": [["pred", "researcher:expertise", ["$r", "$exp"]]],
        "expose": True,
    }
})
print(f"register:  {reg_resp['result']['status']}  "
      f"(total_ephemeral={reg_resp['result']['total_ephemeral']})")

# Session rule inventory — shows FS + ephemeral with source annotation
rules_resp = get_runtime_session_rules(session_id)
for r in rules_resp["result"]["rules"]:
    print(f"  [{r['source']:8s}] {r['rule_id']}@{r['version']}")

In [ ]:
# Evaluate via ruleref to ephemeral rule; explain-steps shows rule name (G1)
eval_eph = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.eph_tag", "version": "v1",
    "target": "researcher:tag", "head_vars": ["$r", "$exp"],
    "where": [["ruleref", "q.eph.expert_tag", "v1", ["$r", "$exp"]]],
    "mode": "native",
}})
print(f"candidates: {eval_eph['meta']['candidate_count']}")

# Candidate inventory — rediscoverable without holding the evaluate response
inv_resp = list_runtime_candidates(session_id)
cid_eph = inv_resp["result"]["candidates"][0]["candidate_id"]
pred_id = inv_resp["result"]["candidates"][0]["pred_id"]
print(f"inventory:  cid={cid_eph[:24]}...  pred_id={pred_id}")

accept_runtime_derivation(session_id, {"candidate": eval_eph["evaluation"]["candidates"][0]})

steps_eph = explain_runtime_steps(session_id, {"kind": "candidate", "id": cid_eph})
print(f"\nexplain-steps ({steps_eph.get('support_kind', '?')}) "
      f"— {len(steps_eph['steps'])} steps")
for s in steps_eph["steps"]:
    indent = "  " * s["detail"]["depth"]
    print(f"  {s['step_num']:2}. {indent}[{s['step_kind']}] {s['description']}")

In [ ]:
# Clear all ephemeral rules; they no longer affect future evaluate calls
clear_resp = clear_ephemeral_rules(session_id)
print(f"clear:     cleared={clear_resp['result']['cleared']}")

# Session rule inventory after clear — only FS rules remain
rules_after = get_runtime_session_rules(session_id)
for r in rules_after["result"]["rules"]:
    print(f"  [{r['source']:8s}] {r['rule_id']}@{r['version']}")

---
## 3. ProbLog — Deep Proof Tree with Probability

Mocked with a 3-level proof trace anchored to the same `Alice` reference used earlier.
This section demonstrates the projected `CandidateEvidenceTree` path for ProbLog:
- intermediate logical frames become `proof_goal`
- terminal logical frames become `proof_leaf`
- probability flows through summary → narrative → NL

### 3.1 Proof Tree — Deep Hierarchy

`proof_goal` (intermediate, has children) vs `proof_leaf` (terminal, no asrt_id).
Shows 3-level proof: tag_seed ← name + expertise.


In [11]:
def _mock_problog_deep():
    a = sdk.ref(Researcher, researcher_id="Alice")
    return "\n".join([
        " call query(X1,X2) {0.00000} []",
        f'  result query(X1,X2) ("senior","{a}") {{{{}}}} {{0.00012}} []',
        " complete query(X1,X2) {0.00013} {0.00013} []",
        f' call answer("senior","{a}") {{0.00019}} [at 4:7]',
        # Level 1: tag_seed goal (has children → proof_goal)
        f'  call researcher__tag_seed("senior","{a}") {{0.00026}} [at 3:9]',
        # Level 2: name goal (leaf → proof_leaf)
        f'   call researcher__name("Alice Chen","{a}") {{0.00032}} [at 2:5]',
        f'    result researcher__name("Alice Chen","{a}") ("Alice Chen","{a}") {{{{}}}} {{0.00040}} [at 2:5]',
        f'   complete researcher__name("Alice Chen","{a}") {{0.00041}} {{0.00009}} []',
        # Level 2: expertise goal (leaf → proof_leaf)
        f'   call researcher__expertise("NLP","{a}") {{0.00045}} [at 2:8]',
        f'    result researcher__expertise("NLP","{a}") ("NLP","{a}") {{{{}}}} {{0.00052}} [at 2:8]',
        f'   complete researcher__expertise("NLP","{a}") {{0.00053}} {{0.00008}} []',
        # Level 1: tag_seed result
        f'   result researcher__tag_seed("senior","{a}") ("senior","{a}") {{{{}}}} {{0.00058}} [at 3:9]',
        f'  complete researcher__tag_seed("senior","{a}") {{0.00059}} {{0.00033}} []',
        # answer result
        f'  result answer("senior","{a}") ("senior","{a}") {{{{}}}} {{0.00065}} []',
        f' complete answer("senior","{a}") {{0.00066}} {{0.00047}} []',
        "",
        f'answer("senior","{a}"):\t0.72',
    ])

with patch("kernel.adapters.problog.engine_eval.run_problog") as mock_run:
    mock_run.return_value = _mock_problog_deep()
    eval_prob = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.problog_tag", "version": "1.0.0",
        "target": "researcher:tag", "head_vars": ["$u", "$tag"],
        "where": [["pred", "researcher:tag_seed", ["$u", "$tag"]]],
        "mode": "problog",
    }})

prob_cand = eval_prob["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": prob_cand})
cid_prob = prob_cand["candidate_id"]
print(f"ProbLog accepted (support_kind={prob_cand['support_kind']})")

ProbLog accepted (support_kind=problog_provenance_v1)


### 3.1 Proof Tree — Deep Hierarchy

`proof_goal` (intermediate, has children) vs `proof_leaf` (terminal, no asrt_id).
Shows 3-level proof: tag_seed ← name + expertise.

In [12]:
tree_prob = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_prob})

print_tree(tree_prob["tree"]["root"])
print(f"probability = {tree_prob['tree']['root']['engine_meta']['probability']}")


- candidate_result | root_result_kind=fact | engine_meta={'engine': 'problog', 'probability': 0.72}
  - support_section
    - proof_goal | pred=researcher__tag_seed | args=['senior', 'idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq']
      - proof_leaf | pred=researcher__name | args=['Alice Chen', 'idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq']
      - proof_leaf | pred=researcher__expertise | args=['NLP', 'idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq']
probability = 0.72


### 3.2 Summary + Narrative + NL

In [13]:
sp = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})["summary"]
np_ = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_prob})["narrative"]
nl_p = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_prob})

print(f"proof_goal_count = {sp.get('proof_goal_count')}")
print(f"proof_leaf_count = {sp.get('proof_leaf_count')}")
print(f"recursive_depth = {sp.get('recursive_depth')}")
print(f"problog_probability = {sp.get('problog_probability')}")
print(f"\nprobability_lines = {np_.get('probability_lines')}")
print(f"evidence_lines = {np_.get('evidence_lines')}")
print(f"rule_chain_lines = {np_.get('rule_chain_lines')}")

print(f"\nNL ({len(nl_p['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_p["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p[:150]}{'...' if len(p)>150 else ''}")

proof_goal_count = 1
proof_leaf_count = 2
recursive_depth = 1
problog_probability = 0.72

probability_lines = ['ProbLog probability: 0.72.']
evidence_lines = ['ProbLog proof tree: 1 intermediate goals, 2 leaf facts.']
rule_chain_lines = ['Proof depth: 1.']

NL (5 paragraphs):
  [1] Candidate cand_v2:da7b96a59785ab4f335c65ecd224aa323853c92c12a0f6b166dbf91cd1fd37af uses support kind problog_provenance_v1 across 5 tree node(s). Root...
  [2] Evidence summary: ProbLog proof tree: 1 intermediate goals, 2 leaf facts.
  [3] Rule-chain summary: Proof depth: 1.
  [4] Terminal and drill-down summary: No unresolved support or recursion boundaries were encountered. Open proof goal nodes to inspect recursive subgoals. ...
  [5] Probability assessment: ProbLog probability: 0.72.


### 3.3 HTML Rendering

In [14]:
html_prob = render_candidate_evidence_html(tree_prob["tree"],
    narrative=explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_prob}).get("narrative"))
print(f"HTML: {len(html_prob)} chars")
display(HTML(f"<div style='border:2px solid #9b59b6;border-radius:8px;overflow:hidden;margin:10px 0'>"
             f"<div style='background:#9b59b6;color:white;padding:6px 12px;font-size:13px'>"
             f"ProbLog — Deep Proof Tree + Probability</div>"
             f"<iframe srcdoc=\'{html_prob.replace(chr(39), '&#39;')}\' "
             f"style='width:100%;height:400px;border:none'></iframe></div>"))

HTML: 16937 chars


---
## 4. PyReason — Multi-Node Graph Propagation

The mocked PyReason trace reuses the same three researcher refs and encodes a simple path:
- `Alice` is the seed chain at `t=0`
- `Bob` is updated from `Alice` at `t=1`
- `Carol` is updated from `Bob` at `t=2`

All prints in the next cells come from `explain_runtime_timeline*()` responses, not from a narrated path string.

### 4.1 Timeline — Multiple Chains + Temporal Propagation


### 3.4 Explain Steps — Proof Path

In [15]:
steps_p = explain_runtime_steps(session_id, {"kind": "candidate", "id": cid_prob})
print(f"ProbLog explain-steps: {len(steps_p['steps'])} steps ({steps_p.get('support_kind', '?')})")
for s in steps_p["steps"]:
    print(f"  {s['step_num']:2}. [{s['step_kind']}] {s['description']}")

ProbLog explain-steps: 5 steps (?)
   1. [proof_leaf_check] Proof leaf: researcher__name("Alice Chen","idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq") (base fact)
   2. [proof_leaf_check] Proof leaf: researcher__expertise("NLP","idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq") (base fact)
   3. [proof_goal_derive] Proof goal: researcher__tag_seed("senior","idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq") derived
   4. [rule_apply] Support group satisfied: 0 condition(s) met
   5. [conclusion] Candidate cand_v2:da7b96a59785ab4f335c65ecd224aa323853c92c12a0f6b166dbf91cd1fd37af established (fact)


In [16]:
derived_session = PyReasonSession(schema_ir)
derived_session._write_node_fact_internal("researcher:risk_flag", alice_ref, "true", bound=[1.0, 1.0])
derived_session._write_node_fact_internal("researcher:risk_flag", bob_ref, "true", bound=[0.8, 0.9])

trace_dict = {
    "engine": "pyreason", "trace_type": "event_log", "timesteps": 3,
    "node_events": [
        # Alice: seed at t=0
        {"time": 0, "fixpoint_op": 1,
         "component": alice_ref, "component_type": "node", "label": "risk_flag",
         "old_bound": [0.0, 1.0], "new_bound": [1.0, 1.0],
         "occurred_due_to": "seed_fact", "clause_groundings": []},
        # Bob: propagated from Alice at t=1
        {"time": 1, "fixpoint_op": 2,
         "component": bob_ref, "component_type": "node", "label": "risk_flag",
         "old_bound": [0.0, 1.0], "new_bound": [0.8, 0.9],
         "occurred_due_to": "risk_propagation",
         "clause_groundings": [f"[{alice_ref}]", f"[({bob_ref}, {alice_ref})]"]},
        # Carol: propagated from Bob at t=2
        {"time": 2, "fixpoint_op": 3,
         "component": carol_ref, "component_type": "node", "label": "risk_flag",
         "old_bound": [0.0, 1.0], "new_bound": [0.7, 0.85],
         "occurred_due_to": "risk_propagation",
         "clause_groundings": [f"[{bob_ref}]", f"[({carol_ref}, {bob_ref})]"]},
    ],
    "edge_events": [],
}

with patch("kernel.adapters.pyreason.engine_eval.run_pyreason") as mock_pr:
    mock_pr.return_value = PyReasonRunResult(
        interpretation=None, trace=None, trace_dict=trace_dict,
        derived_session=derived_session,
        config=PyReasonRunConfig(timesteps=3, atom_trace=True),
        elapsed_seconds=0.02,
    )
    eval_pr = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.pyreason_risk", "version": "1.0.0",
        "target": "researcher:risk_flag", "head_vars": ["$r"],
        "where": [["pred", "researcher:expertise", ["$r", "$exp"]]],
        "mode": "pyreason",
    }})

pr_cand = eval_pr["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": pr_cand})
cid_pr = pr_cand["candidate_id"]
print(
    f"PyReason accepted: support_kind={pr_cand['support_kind']}, "
    f"timesteps={trace_dict['timesteps']}, node_events={len(trace_dict['node_events'])}"
)


PyReason accepted: support_kind=pyreason_provenance_v1, timesteps=3, node_events=3


### 4.1 Timeline — Multiple Chains + Temporal Propagation

In [17]:
tl = explain_runtime_timeline(session_id, {"kind": "candidate", "id": cid_pr})
s_pr = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})
n_pr = explain_runtime_timeline_narrative(session_id, {"kind": "candidate", "id": cid_pr})
nl_pr = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_pr})

timeline = tl["timeline"]
root_chain_key = tuple(timeline["root_chain_key"])
print(f"timesteps: {timeline['timesteps']}, chains: {len(timeline['chains'])}")
for chain in timeline["chains"]:
    comp = chain["component"].split(":")[-1][:20]
    chain_key = (chain["component_type"], chain["component"], chain["label"])
    print(f"\n  Chain: ...{comp}.{chain['label']}" + (" [ROOT]" if chain_key == root_chain_key else ""))
    for evt in chain["events"]:
        print(f"    t={evt['time']}: {evt['old_bound']} → {evt['new_bound']} by {evt['trigger']}")
        if evt.get("groundings"):
            for g in evt["groundings"]:
                print(f"           grounding: {g[:60]}")

print(
    f"\nSummary: chains={s_pr['summary']['chain_count']}, "
    f"seeds={s_pr['summary']['seed_count']}, "
    f"derived={s_pr['summary']['derived_count']}"
)

print(f"\nNarrative headline: {n_pr['narrative']['headline']}")
for line in n_pr["narrative"].get("propagation_lines", []):
    print(f"  {line[:120]}")

print(f"\nNL ({len(nl_pr['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_pr["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p[:120]}{'...' if len(p)>120 else ''}")


timesteps: 3, chains: 3

  Chain: ...2liyiecqxqizhtnfb45w.risk_flag
    t=2: [0.0, 1.0] → [0.7, 0.85] by risk_propagation
           grounding: [idref_v1:Researcher:dlvr3hixzzjzgnqyhp4v2uncoaftdfnlj2km74g
           grounding: [(idref_v1:Researcher:2liyiecqxqizhtnfb45wmxxl5ydukvvvkv3feh

  Chain: ...5bglgazrk2y34euilrjy.risk_flag [ROOT]
    t=0: [0.0, 1.0] → [1.0, 1.0] by seed_fact

  Chain: ...dlvr3hixzzjzgnqyhp4v.risk_flag
    t=1: [0.0, 1.0] → [0.8, 0.9] by risk_propagation
           grounding: [idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75f
           grounding: [(idref_v1:Researcher:dlvr3hixzzjzgnqyhp4v2uncoaftdfnlj2km74

Summary: chains=3, seeds=1, derived=2

Narrative headline: risk_flag: 3 chains, 3 timesteps
  t=0: idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq.risk_flag seeded [1.0, 1.0]
  t=1: idref_v1:Researcher:dlvr3hixzzjzgnqyhp4v2uncoaftdfnlj2km74gjqjwseunebyxa.risk_flag updated to [0.8, 0.9] by risk_pr
  t=2: idref_v1:Researcher:2

### 4.2 HTML — Timeline Rendering

In [18]:
from kernel.adapters.pyreason.provenance import pyreason_trace_to_evidence_graph, pyreason_trace_from_dict

eg_timeline = pyreason_trace_to_evidence_graph(
    pyreason_trace_from_dict(trace_dict),
    candidate_id=cid_pr,
    candidate_payload=pr_cand.get("payload", {}),
)

html_pr = render_evidence_graph_html(eg_timeline)
print(f"EvidenceGraph: layout={eg_timeline.layout_hint}, nodes={len(eg_timeline.nodes)}, edges={len(eg_timeline.edges)}")
display(HTML(f"<div style='border:2px solid #27ae60;border-radius:8px;overflow:hidden;margin:10px 0'>"
             f"<div style='background:#27ae60;color:white;padding:6px 12px;font-size:13px'>"
             f"PyReason — Multi-Node Timeline Propagation</div>"
             f"<iframe srcdoc=\'{html_pr.replace(chr(39), '&#39;')}\' "
             f"style='width:100%;height:350px;border:none'></iframe></div>"))

EvidenceGraph: layout=timeline, nodes=3, edges=0


---
## 5. Cross-Engine Comparison

The final cell prints only summary scalars returned by the explain endpoints.
It is meant to show how the same session exposes three different explain carriers:
- native / Souffle style evidence tree with certainty
- ProbLog proof tree with probability
- PyReason provenance timeline with seed / derived chain counts


### 4.3 Explain Steps — Timeline as Steps

In [19]:
steps_pr = explain_runtime_steps(session_id, {"kind": "candidate", "id": cid_pr})
print(f"PyReason explain-steps: {len(steps_pr['steps'])} steps ({steps_pr.get('support_kind', '?')})")
for s in steps_pr["steps"]:
    print(f"  {s['step_num']:2}. [{s['step_kind']:<18}] {s['description']}")

PyReason explain-steps: 4 steps (?)
   1. [bound_seed        ] t=0: idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq.risk_flag initialized to [1.000, 1.000]
   2. [bound_update      ] t=1: idref_v1:Researcher:dlvr3hixzzjzgnqyhp4v2uncoaftdfnlj2km74gjqjwseunebyxa.risk_flag updated [0.000, 1.000] -> [0.800, 0.900] by risk_propagation
   3. [bound_update      ] t=2: idref_v1:Researcher:2liyiecqxqizhtnfb45wmxxl5ydukvvvkv3fehkr34wcq6ybdboa.risk_flag updated [0.000, 1.000] -> [0.700, 0.850] by risk_propagation
   4. [convergence       ] Converged after 3 timestep(s) across 3 chain(s). Final bounds: idref_v1:Researcher:2liyiecqxqizhtnfb45wmxxl5ydukvvvkv3fehkr34wcq6ybdboa.risk_flag=[0.700, 0.850]; idref_v1:Researcher:5bglgazrk2y34euilrjyq24w4x74vamid7qf75fek32yw4hkfnrq.risk_flag=[1.000, 1.000]; idref_v1:Researcher:dlvr3hixzzjzgnqyhp4v2uncoaftdfnlj2km74gjqjwseunebyxa.risk_flag=[0.800, 0.900]


In [20]:
sn = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
cs = sn.get("certainty_summary")
print(f"[Native] witnesses={sn['summary']['witness_assertion_count']}, "
      f"rule_refs={sn['summary']['rule_ref_count']}, "
      f"recursive_depth={sn['summary']['recursive_depth']}")
if cs:
    print(f"  certainty={cs['aggregate_certainty']} ({cs['aggregation']})")

sp = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})["summary"]
print(f"[ProbLog] proof_goals={sp.get('proof_goal_count', 0)}, "
      f"proof_leaves={sp.get('proof_leaf_count', 0)}, depth={sp.get('recursive_depth', 0)}")
print(f"  probability={sp.get('problog_probability')}")

st = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})["summary"]
print(f"[PyReason] chains={st['chain_count']}, seeds={st['seed_count']}, "
      f"derived={st['derived_count']}, timesteps={st['timesteps']}")


[Native] witnesses=4, rule_refs=2, recursive_depth=2
[ProbLog] proof_goals=1, proof_leaves=2, depth=1
  probability=0.72
[PyReason] chains=3, seeds=1, derived=2, timesteps=3


In [21]:
close_resp = close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
print(close_resp["closed"])


{'session_id': 'rt_30200e3a7c314ca5bafe28b73a5c6588'}


## Architecture Summary

```
  Shared Schema (Researcher + Collaboration) + Store (one session, one ledger)
       │
       ├── Native evaluate → accept → CandidateEvidenceTree
       │     Multi-rule chain: grant_qualification → established_check
       │     predicate_witness_group / assertion_fact + certainty (bottleneck/additive)
       │     recursive_depth > 0, condition_weights flow through RuleRef chains
       │     explain-steps → [fact_check, rule_apply, conclusion]; source_lines from fact_meta
       │
       ├── ProbLog evaluate → accept → CandidateEvidenceTree
       │     Deep proof tree: proof_goal (intermediate) → proof_leaf (terminal)
       │     problog_probability flows: summary → narrative → NL
       │     proof_leaf has no asrt_id (no dead links in static UI)
       │     explain-steps → [proof_leaf_check, proof_goal_derive, conclusion]
       │
       └── PyReason evaluate → accept → CandidateProvenanceTimeline
             Multi-node graph: Alice → Bob → Carol
             Bound propagation across timesteps [1.0,1.0] → [0.8,0.9] → [0.7,0.85]
             explain-timeline endpoints + polymorphic NL dispatch
             EvidenceGraph(timeline) for audit rendering
             explain-steps → [bound_seed, bound_update, convergence]

  Cross-engine: accepted facts from Engine A visible to Engine B
  Unified interface, not unified implementation
```